Configure Source Paths


In [0]:

CATALOG = "capretailnew"
SCHEMA = "retail"
VOLUME = "ext_retail_vol"

ROOT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

SALES_PATH = ROOT_PATH + "/sales_order_lines"
INVENTORY_PATH = ROOT_PATH + "/inventory_movements"
RETURNS_PATH = ROOT_PATH + "/sales_returns"
display(dbutils.fs.ls(ROOT_PATH))

Chech Source Folders

In [0]:
print("Sales Files")
display(dbutils.fs.ls(SALES_PATH))

print("Inventory Files")
display(dbutils.fs.ls(INVENTORY_PATH))

print("Returns Files")
display(dbutils.fs.ls(RETURNS_PATH))

Create Required folders

In [0]:
folders = [
    "checkpoints/sales",
    "checkpoints/inventory",
    "checkpoints/returns",
    "schema_tracking/sales",
    "schema_tracking/inventory",
    "schema_tracking/returns",
    "logs",
    "quarantine"
]

for folder in folders:
    dbutils.fs.mkdirs(ROOT_PATH + "/" + folder)

print("Required folders created successfully")

In [0]:
# Import Required Libraries
from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    lit,
    sum,
    min,
    max,
    avg
)

# Read Sales CSV Files

sales_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(SALES_PATH)
)

print("Sales Record Count:", sales_df.count())

display(sales_df)

In [0]:
# DBTITLE 1,6. Check Sales Schema

sales_df.printSchema()

In [0]:
#  Check Sales Columns

print("Sales Columns:")
print(sales_df.columns)

In [0]:
#  Check Sales Missing Values

sales_missing = sales_df.select([
    count(
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            c
        )
    ).alias(c)
    for c in sales_df.columns
])

display(sales_missing)

In [0]:
# Check Duplicate Sales Records
print("Total Sales Records:", sales_df.count())

print(
    "Distinct Order Line IDs:",
    sales_df.select("order_line_id").distinct().count()
)

duplicate_sales = (
    sales_df.groupBy("order_line_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate Order Line IDs:")
display(duplicate_sales)


In [0]:
# DBTITLE 1,10. Check Sales Data Distribution
display(
    sales_df.groupBy("order_status")
    .count()
)

In [0]:
# DBTITLE 1,11. Check Sales Numeric Columns
sales_numeric_df = (
    sales_df
    .withColumn("quantity_num", col("quantity").cast("double"))
    .withColumn("unit_price_num", col("unit_price").cast("double"))
    .withColumn("discount_pct_num", col("discount_pct").cast("double"))
)

display(
    sales_numeric_df.select(
        "quantity_num",
        "unit_price_num",
        "discount_pct_num"
    ).summary()
)

In [0]:
#  Read Inventory Data
inventory_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(INVENTORY_PATH)
)

print("Inventory Record Count:", inventory_df.count())

display(inventory_df)

In [0]:
# DBTITLE 1,13. Check Inventory Schema
inventory_df.printSchema()

In [0]:
# DBTITLE 1,14. Check Inventory Missing Values
inventory_missing = inventory_df.select([
    count(
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            c
        )
    ).alias(c)
    for c in inventory_df.columns
])

display(inventory_missing)

In [0]:
# DBTITLE 1,15. Check Inventory Duplicate Records

print("Total Inventory Records:", inventory_df.count())

print(
    "Distinct Movement IDs:",
    inventory_df.select("movement_id").distinct().count()
)

duplicate_inventory = (
    inventory_df.groupBy("movement_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_inventory)

In [0]:
# DBTITLE 1,16. Check Inventory Movement Types
display(
    inventory_df.groupBy("movement_type")
    .count()
)

In [0]:
# DBTITLE 1,17. Read Returns Data

returns_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(RETURNS_PATH)
)

print("Returns Record Count:", returns_df.count())

display(returns_df)

In [0]:
# DBTITLE 1,18. Check Returns Schema

returns_df.printSchema()

In [0]:
# DBTITLE 1,19. Check Returns Missing Values

returns_missing = returns_df.select([
    count(
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            c
        )
    ).alias(c)
    for c in returns_df.columns
])

display(returns_missing)

In [0]:
# COMMAND ----------

# DBTITLE 1,20. Check Returns Duplicate Records

print("Total Returns Records:", returns_df.count())

print(
    "Distinct Return IDs:",
    returns_df.select("return_id").distinct().count()
)

duplicate_returns = (
    returns_df.groupBy("return_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_returns)




In [0]:
from pyspark.sql.functions import col

df = spark.table("capretailnew.bronze.bronze_sales_order_lines")

display(df)



In [0]:
from pyspark.sql.functions import col

def find_outliers(df, column_name):

    # Calculate Q1 and Q3
    q1, q3 = df.approxQuantile(
        column_name, [0.25, 0.75], 0.01
    )

    # Calculate IQR
    iqr = q3 - q1

    # Calculate boundaries
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    # Identify outliers
    outliers_df = df.filter(
        (col(column_name) < lower_bound) |
        (col(column_name) > upper_bound)
    )

    print("Column:", column_name)
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Outlier Count:", outliers_df.count())

    display(outliers_df)

    return outliers_df

In [0]:
numeric_columns = [
    "quantity",
    "unit_price",
    "discount_pct"
]

for column_name in numeric_columns:
    find_outliers(sales_df, column_name)

In [0]:
# COMMAND ----------

# DBTITLE 1,21. Check Return Reasons

display(
    returns_df.groupBy("return_reason")
    .count()
)


# COMMAND ----------

# DBTITLE 1,22. Check Return Inspection Status

display(
    returns_df.groupBy("inspection_status")
    .count()
)


# COMMAND ----------

# DBTITLE 1,23. Source Data Summary

print("SALES COUNT:", sales_df.count())
print("INVENTORY COUNT:", inventory_df.count())
print("RETURNS COUNT:", returns_df.count())

print("Source Data Profiling Completed")